# 07 · Acceptance rate per condition, from the measurement chain (supplementary)

A capture is **accepted** when it decodes to `lewb`, its label coverage is ≥ 95 % of the norm label, and the in-plane
rotation is ≤ 2°. Rejected captures are retaken, not mis-measured, so the acceptance rate is the recapture cost.

This is an approximation of the app's gates built from the measurement chain of notebook 06 (coverage by area instead of
the app's four-corner check). The paper's Table 2 reports the **app's own** decision on each frame, which is computed in
notebook 14 and is stricter for pitch (2 of 11 accepted there, 5 of 11 here).

| | |
|---|---|
| **Inputs** | `data/raw/robustness/` |
| **Outputs** | `results/tables/acceptance.csv`, `acceptance.tex` |
| **Runtime** | ~15 s |

> **Folder names.** The capture folders were renamed after the original analysis. Old → current:
> `angle → roll` (in-plane rotation, measured ≈15°) and `roll → pitch` (out-of-plane tilt).
> `metadata.csv` keeps the old name in `legacy_name`. The three numbers in each file name
> (`pose_a/b/c`) are the phone's orientation sensor readings at capture time, **not** the label's pose.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import visioncrack as vc
from visioncrack import io
np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.width', 200)
print('visioncrack', vc.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)
FIG = io.results_dir('figures'); TAB = io.results_dir('tables')

visioncrack 0.1.0 | numpy 2.2.6 | pandas 2.3.0


In [ ]:
from visioncrack import sweep
res, ref = sweep.run_all()          # ~10 s for 45 captures
FOLDERS = sweep.FOLDERS
print('reference (norm) label box cx, cy, w, h =', np.round(ref.box, 1))
res.groupby('folder', sort=False).size().rename('captures').to_frame().T

reference (norm) label box cx, cy, w, h = [1509.  3129.   854.5  500.5]


folder,norm,dist,roll,pitch,yaw
captures,4,6,12,11,12


In [ ]:
res['inplane_ok'] = res.inplane_deg.abs() <= 2.0
res['accepted'] = res.decode_ok & res.coverage_ok & res.inplane_ok
def agg(g):
    meas = g[g.ref_found]
    return pd.Series({'n': len(g), 'decode_%': 100 * g.decode_ok.mean(), 'coverage_ge95_%': 100 * g.coverage_ok.mean(),
                      'inplane_le2_%': 100 * g.inplane_ok.mean(), 'accepted_%': 100 * g.accepted.mean(),
                      'mean_coverage': g.coverage.mean(),
                      'mean_MAE_mm': meas.ref_mae.mean() if len(meas) else np.nan})
acc = res.groupby('folder').apply(agg, include_groups=False).reindex(FOLDERS)
acc.loc['All'] = agg(res)
acc.to_csv(TAB / 'acceptance.csv', float_format='%.4f')
acc.round(3)

,n,decode_%,coverage_ge95_%,inplane_le2_%,accepted_%,mean_coverage,mean_MAE_mm
folder,,,,,,,
norm,4.0,100.000,100.000,100.000,100.000,1.000,0.061
dist,6.0,100.000,0.000,100.000,0.000,0.381,0.074
roll,12.0,100.000,0.000,0.000,0.000,0.665,0.197
pitch,11.0,100.000,45.455,100.000,45.455,0.882,NaN
yaw,12.0,50.000,25.000,58.333,0.000,0.926,NaN
All,45.0,86.667,26.667,62.222,20.000,0.779,0.139


### Regression check
Expected values, in the current folder order (norm, dist, roll, pitch, yaw). The run stops if a result changes.

In [ ]:
expected = pd.DataFrame({
    'decode_%':        [100, 100, 100, 100, 50, 87],
    'coverage_ge95_%': [100, 0, 0, 45, 25, 27],
    'inplane_le2_%':   [100, 100, 0, 100, 58, 62],
    'accepted_%':      [100, 0, 0, 45, 0, 20],
    'mean_coverage':   [1.00, 0.38, 0.67, 0.88, 0.93, 0.78],
}, index=FOLDERS + ['All'])
assert np.allclose(acc[expected.columns[:4]].round(0), expected[expected.columns[:4]])
assert np.allclose(acc['mean_coverage'].round(2), expected['mean_coverage'])
assert np.allclose(acc['mean_MAE_mm'].round(3).dropna(), [0.061, 0.074, 0.197, 0.139])
print('✓ matches the expected values')

✓ matches the expected values


### Which folder is which kind of tilt
The old folder names suggest the opposite of what the photos show:

* old `angle` (now `roll/`): measured **in-plane** rotation ≈ 15°, so 0 % pass the 2° gate;
* old `roll` (now `pitch/`): in-plane rotation < 1°, coverage 0.88, i.e. **out-of-plane** tilt.

The LaTeX table below uses the current names.

In [ ]:
res.groupby('folder')[['inplane_deg', 'coverage']].mean().reindex(FOLDERS).round(2)

,inplane_deg,coverage
folder,,
norm,0.44,1.00
dist,0.45,0.38
roll,15.04,0.67
pitch,0.87,0.88
yaw,1.81,0.93


In [ ]:
LABEL = {'norm': r'\emph{norm} (reference pose)', 'dist': r'\emph{dist} (far / low cov.)',
         'roll': r'\emph{roll} (in-plane rotation)', 'pitch': r'\emph{pitch} (out-of-plane tilt)',
         'yaw': r'\emph{yaw} (out-of-plane yaw)', 'All': r'\textbf{All}'}
def row(k, r):
    mae = '--$^{\\dagger}$' if pd.isna(r['mean_MAE_mm']) else f"{r['mean_MAE_mm']:.3f}"
    return (f"{LABEL[k]} & {int(r['n'])} & {r['decode_%']:.0f} & {r['coverage_ge95_%']:.0f} & "
            f"{r['inplane_le2_%']:.0f} & \\textbf{{{r['accepted_%']:.0f}}} & {r['mean_coverage']:.2f} & {mae} \\\\")
lines = [row(k, acc.loc[k]) for k in FOLDERS] + ['\\midrule', row('All', acc.loc['All'])]
(TAB / 'acceptance.tex').write_text('\n'.join(lines))
print((TAB / 'acceptance.tex').read_text())

\emph{norm} (reference pose) & 4 & 100 & 100 & 100 & \textbf{100} & 1.00 & 0.061 \\
\emph{dist} (far / low cov.) & 6 & 100 & 0 & 100 & \textbf{0} & 0.38 & 0.074 \\
\emph{roll} (in-plane rotation) & 12 & 100 & 0 & 0 & \textbf{0} & 0.67 & 0.197 \\
\emph{pitch} (out-of-plane tilt) & 11 & 100 & 45 & 100 & \textbf{45} & 0.88 & --$^{\dagger}$ \\
\emph{yaw} (out-of-plane yaw) & 12 & 50 & 25 & 58 & \textbf{0} & 0.93 & --$^{\dagger}$ \\
\midrule
\textbf{All} & 45 & 87 & 27 & 62 & \textbf{20} & 0.78 & 0.139 \\
